# Naive Bayes — Titanic Dataset

Klasifikasi `Survived` menggunakan fitur `Age` dan `Fare` dengan Gaussian Naive Bayes.
Alur: load → cleaning → impute → split → scale → fit → predict → evaluate → visualize.

## 1. Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

## 2. Importing and cleaning the dataset

In [ ]:
dataset = pd.read_csv('titanic.csv')
print('Shape awal:', dataset.shape)
print(dataset.dtypes)
dataset.head()

In [ ]:
# Drop kolom yang tidak dipakai (banyak string / tidak relevan untuk fitur 2D)
dataset = dataset.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Encode kolom kategorikal -> angka
dataset['Sex']      = dataset['Sex'].map({'male': 0, 'female': 1})
dataset['Embarked'] = dataset['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Sanity check: semua kolom harus numerik (boleh ada NaN, akan di-impute)
print(dataset.dtypes)
print('Kolom object tersisa:', dataset.select_dtypes(include='object').columns.tolist())

## 3. Selecting features & target

In [ ]:
# Fitur 2D untuk visualisasi decision boundary
X = dataset[['Age', 'Fare']].values
y = dataset['Survived'].values

print('X shape:', X.shape)
print('y shape:', y.shape)
print('NaN di X (sebelum impute):', np.isnan(X).sum())
print(X[:5])
print(y[:5])

## 4. Imputing missing values (manual — sebelum split)

GaussianNB tidak menerima NaN. `Age` dan `Fare` pada Titanic punya nilai kosong,
jadi kita isi dengan median. Ini juga mencegah error `Input X contains NaN` di Step fit.

In [ ]:
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

print('NaN di X (setelah impute):', np.isnan(X).sum())   # harus 0
assert np.isnan(X).sum() == 0, 'Masih ada NaN di X!'

## 5. Splitting the dataset into the Training set and Test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

## 6. Feature Scaling

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

print('X_train[:3]:')
print(X_train[:3])
print('X_test[:3]:')
print(X_test[:3])

## 7. Training the Naive Bayes model

In [ ]:
classifier = GaussianNB()
classifier.fit(X_train, y_train)

# Verifikasi: pastikan bukan pipeline dan sudah fitted
from sklearn.utils.validation import check_is_fitted
check_is_fitted(classifier)
print('Classifier type :', type(classifier).__name__)
print('Classifier siap.' )

## 8. Predicting the Test set results

In [ ]:
y_pred = classifier.predict(X_test)

print('y_pred[:10]:', y_pred[:10])
print('y_test[:10]:', y_test[:10])
print('Accuracy   :', accuracy_score(y_test, y_pred))

## 9. Confusion Matrix & Classification Report

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

f, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set_xlabel('y_pred')
ax.set_ylabel('y_true')
ax.set_title('Confusion Matrix — Titanic / GaussianNB')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Not Survived (0)', 'Survived (1)']))

## 10. Visualising the Training set results

In [ ]:
def plot_decision_boundary(X_set, y_set, classifier, title):
    X1, X2 = np.meshgrid(
        np.arange(start=X_set[:, 0].min() - 1, stop=X_set[:, 0].max() + 1, step=0.01),
        np.arange(start=X_set[:, 1].min() - 1, stop=X_set[:, 1].max() + 1, step=0.01)
    )
    plt.contourf(
        X1, X2,
        classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
        alpha=0.75, cmap=ListedColormap(('red', 'green'))
    )
    plt.xlim(X1.min(), X1.max())
    plt.ylim(X2.min(), X2.max())
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(
            X_set[y_set == j, 0], X_set[y_set == j, 1],
            color=ListedColormap(('red', 'green'))(i),
            label=('Not Survived' if j == 0 else 'Survived')
        )
    plt.title(title)
    plt.xlabel('Age (scaled)')
    plt.ylabel('Fare (scaled)')
    plt.legend()
    plt.show()

plot_decision_boundary(X_train, y_train, classifier, 'Naive Bayes (Training set) — Titanic')

## 11. Visualising the Test set results

In [ ]:
plot_decision_boundary(X_test, y_test, classifier, 'Naive Bayes (Test set) — Titanic')

## 12. Final sanity check

In [ ]:
print('NaN di X_train :', np.isnan(X_train).sum())
print('NaN di X_test  :', np.isnan(X_test).sum())
print('NaN di y_train :', np.isnan(y_train).sum())
print('Accuracy       :', accuracy_score(y_test, y_pred))

assert np.isnan(X_train).sum() == 0
assert np.isnan(X_test).sum() == 0
assert np.isnan(y_train).sum() == 0
print('\nSemua cek lulus. Notebook siap.')

# Simpan hasil prediksi (opsional)
pd.DataFrame({'y_true': y_test, 'y_pred': y_pred}).to_csv(
    'titanic_nb_predictions.csv', index=False)
print('Prediksi disimpan ke titanic_nb_predictions.csv')